# Module 02 — Operators

Arithmetic, comparison, logic, and bit masks. Assumes module 01.

Most of this behaves the way you expect. The sections below are the ones where it
does not.

## 1. Precedence

Strongest to weakest: `**` · `-x` · `* / // %` · `+ -` · comparisons ·
`not` → `and` → `or`.

Two entries hide something. **Predict all three:**

In [ ]:
assert 2 + 3 * 4 == ...
assert 2**3**2 == ...
assert -(3**2) == ...

**`**` groups from the right.** `2 ** 3 ** 2` is `2 ** (3 ** 2)` = `2 ** 9`. Every
other arithmetic operator in Python groups from the left, and so does every
operator you know from C or Java — neither language has `**` at all, so there is
no habit to fall back on.

**`**` binds tighter than unary minus.** `-3 ** 2` is `-(3 ** 2)`. For "minus
three, squared", write `(-3) ** 2`.

## 2. The sign of a remainder

`-17 % 5` is `3` in Python and `-2` in C and Java. The result takes the sign of
the **right** operand here, of the left one there.

This is the one place where porting an arithmetic expression between the
languages silently changes its meaning — and it bites hardest in the idiom `%` is
most used for: wrapping an index back into range.

In [ ]:
assert 17 % 5 == ...
assert -17 % 5 == ...

In [ ]:
# Wrapping an index: always lands inside the list, whatever the offset.
items = ["a", "b", "c"]
for offset in (-4, -1, 0, 4):
    print(offset, "->", items[offset % len(items)])

## 3. Chained comparison

`1 < x < 10` is one expression meaning both comparisons, with `x` evaluated once.
In C and Java the same line parses as `(1 < x) < 10` and quietly computes
something useless.

Python has both readings. **Predict the difference:**

In [ ]:
assert (3 > 2 > 1) is ...
assert ((3 > 2) > 1) is ...

The bracketed form evaluates `3 > 2` to `True`, then asks `True > 1` — and `True`
is `1` in arithmetic, so that is `1 > 1`.

Use the chain. `-40 <= reading <= 85` is the idiomatic range check.

In [ ]:
reading = 21.7
print(-40 <= reading <= 85)
print(True + True, int(True))

## 4. What `and` and `or` return

The truth values hold no surprises. The **return value** does: `and` and `or` do
not produce `True` or `False`, they produce the operand that settled the question.
`&&` and `||` in C and Java always give you a boolean, so this has no counterpart
there.

In [ ]:
assert (0 or "empty") == ...
assert ("a" and "b") == ...

`0` is falsy, so `or` moves on and hands back `"empty"`. `"a"` is truthy, so `and`
has to evaluate the right side and hands that back.

Hence the default idiom, which stands in for `?:`:

```python
name = user_input or "unknown"
```

With one sharp edge: `count or 10` also falls back when `count` is `0`, because
zero is falsy. Where zero is a legitimate value, test `is None` explicitly.

## 5. Short-circuit evaluation

Identical to `&&` and `||`: the right operand is not evaluated once the answer is
settled. As there, it is a guarantee of the language and you are meant to order
your guards around it.

In [ ]:
total = 0
count = 0

# The left operand keeps the division from ever happening.
print(count != 0 and total / count > 10)

## 6. `==` against `is`

This is Java's `==` against `.equals()`, with the names swapped:

| Python | Java | asks |
|---|---|---|
| `is` | `==` | same object? |
| `==` | `.equals()` | same value? |

Reading Python with Java reflexes gets it exactly the wrong way round, which is
why this is the most common confusion in the language.

In [ ]:
a = [1, 2]
b = [1, 2]

assert (a == b) is ...
assert (a is b) is ...

**Use `==` for values.** `is` has one everyday use — `if result is None` — which
works because there is exactly one `None` object in a running program.

Do not use `is` on numbers or short strings even when it appears to work. Whether
two equal values happen to be one object is an implementation decision, not a
promise. It is also observable, which is what makes it tempting:

In [ ]:
a = 1000
b = 1000
print(a == b, a is b)

# Now put those same three lines in a .py file and run it with uv run.
# The answer changes -- which is the point: it is not something to rely on.

## 7. Bit masks

The operators are the ones you know from C: `&`, `|`, `^`, `~`, `<<`, `>>`, doing
the same job on the same idiom. A device reports one byte, each bit a flag:

```
0 0 0 0 0 1 1 0
              └─ bit 0: measurement ready
            └─── bit 1: limit exceeded
          └───── bit 2: sensor fault
        └─────── bit 3: calibration due
```

Read a bit with `&`, set it with `|`, clear it with `& ~`, toggle it with `^`.
Binary literals are `0b`, and `bin()` prints them back.

In [ ]:
READY, LIMIT, FAULT, CALIBRATE = 0b0001, 0b0010, 0b0100, 0b1000
status = 0b0110

# What survives after masking away everything but the LIMIT bit?
assert (status & LIMIT) == ...

# And READY, which is not set?
assert (status & READY) == ...

`status & LIMIT` gives the bit **in its own column** — 2, not 1. For a yes/no
answer, wrap it: `bool(status & LIMIT)`.

In [ ]:
READY, LIMIT, FAULT, CALIBRATE = 0b0001, 0b0010, 0b0100, 0b1000
status = 0b0110

print(bool(status & LIMIT))
print(bin(status | READY))  # set READY
print(bin(status & ~FAULT))  # clear FAULT

### One difference from C, and it matters here

In C, `~` complements within the width of the type: `~4` on a `uint8_t` is `251`.
Python integers have no width, so `~x` is `-x - 1` and stays negative.

In [ ]:
print(~4)  # -5, not 251
print(bin(~4))  # -0b101
print(bin(~4 & 0xFF))  # 0b11111011 -- masked back to a width you chose

When you need a byte-shaped result — writing to a register, building a protocol
frame — say so: `~x & 0xFF`. The same applies to `<<`: nothing shifts off the top,
because there is no top.

**Where you meet this:** GPIO pins, status registers in datasheets, file
permissions (`chmod 755`), protocol flags.

---

On to `exercises/`, then `uv run pytest 02_operators`.